In [ ]:
!pip install streamlit pyngrok streamlit-option-menu bcrypt PyJWT -q

In [ ]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, re, streamlit as st
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
# --- 1. INITIAL CONFIG & STATE ---
st.set_page_config(page_title="Intelligent Freight Hub", page_icon="🚢", layout="wide", initial_sidebar_state="expanded")
if "theme" not in st.session_state: st.session_state.theme = "dark"
for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None), ("jwt_otp_token", ""), ("last_otp_time", 0)]:
    if k not in st.session_state: st.session_state[k] = v
def navigate(p): st.session_state.page = p; st.rerun()
# --- 2. RESTRICTED SIDEBAR (DASHBOARD ONLY) ---
if st.session_state.token:
    with st.sidebar:
        st.markdown("<h2 style='text-align:center;'>⚙️ Settings</h2>", unsafe_allow_html=True)
        is_dark = st.toggle("🌙 Dark Mode", value=(st.session_state.theme == "dark"))
        st.session_state.theme = "dark" if is_dark else "light"
        st.markdown("<hr style='margin:10px 0; border-color: gray;'>", unsafe_allow_html=True)
        if st.button("🚪 Logout", use_container_width=True):
            st.session_state.token = None
            navigate("Login")
else:
    st.markdown("""<style>[data-testid="collapsedControl"] {display: none;} [data-testid="stSidebar"] {display: none;}</style>""", unsafe_allow_html=True)
# --- 3. DYNAMIC COLOR PALETTE ---
if st.session_state.theme == "dark":
    COLORS = {"bg": "#0F172A", "card": "#1E293B", "text": "#F8FAFC", "muted": "#94A3B8", "accent": "#EAB308", "border": "#334155", "input_bg": "#0B1120"}
else:
    COLORS = {"bg": "#F0F4F8", "card": "#FFFFFF", "text": "#0F172A", "muted": "#64748B", "accent": "#D97706", "border": "#CBD5E1", "input_bg": "#F8FAFC"}
JWT_SECRET = os.environ.get("JWT_SECRET", "fallback-secret-key")
SENDER_EMAIL = os.environ.get("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD", "")
OTP_EXPIRY_MINUTES = 5
# --- 4. SAFE & STABLE CSS ---
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
    html, body, .stApp {{ font-family: 'Inter', sans-serif !important; background-color: {COLORS['bg']} !important; color: {COLORS['text']} !important; }}
    .stApp {{ background-image: radial-gradient({COLORS['border']} 1px, transparent 1px) !important; background-size: 24px 24px !important; }}
    header, footer, div[data-testid="stDecoration"] {{ display: none !important; }}
    .block-container {{ padding: 3rem 2rem !important; max-width: 900px; }}
    div[data-testid="stVerticalBlock"] > div[data-testid="stVerticalBlock"] {{ background-color: {COLORS['card']} !important; border: 1px solid {COLORS['border']} !important; border-radius: 8px !important; padding: 32px !important; box-shadow: 0 4px 6px rgba(0,0,0,0.05) !important; }}
    p, label, span, h1, h2, h3 {{ color: {COLORS['text']} !important; }}
    div[data-baseweb="input"] {{ background-color: {COLORS['input_bg']} !important; border: 1px solid {COLORS['border']} !important; border-radius: 6px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; }}
    input {{ color: {COLORS['text']} !important; -webkit-text-fill-color: {COLORS['text']} !important; }}
    input::placeholder {{ color: {COLORS['muted']} !important; -webkit-text-fill-color: {COLORS['muted']} !important; opacity: 1 !important; }}
    div[data-testid="InputInstructions"] {{ display: none !important; visibility: hidden !important; }}
    div[data-testid="stButton"] button {{ background-color: {COLORS['accent']} !important; color: #111827 !important; border: none !important; border-radius: 6px !important; font-weight: 600 !important; height: 44px !important; transition: transform 0.1s ease !important; }}
    div[data-testid="stButton"] button:hover {{ filter: brightness(1.1); transform: translateY(-1px); }}
</style>
""", unsafe_allow_html=True)
# --- 5. DATABASE HELPERS ---
# [FIX #8] Added WAL mode for safe concurrent access under Streamlit's threaded reruns.
def get_db():
    conn = sqlite3.connect("infosys_portal.db", check_same_thread=False)
    conn.execute("PRAGMA journal_mode=WAL")
    return conn
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False
with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='admin@infosys.com'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", ("Administrator", "admin@infosys.com", hash_txt("Admin@123!"), "What is your pet name?", hash_txt("admin")))
    conn.commit()
# [FIX #2 & #3] Replaced deprecated utcnow() with timezone-aware now(UTC).
def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None
# --- 6. DETAILED VALIDATION ENGINES ---
def get_password_error(pwd):
    if len(pwd) < 8: return "Password must be at least 8 characters long."
    if not re.search(r"[A-Z]", pwd): return "Password must contain at least one uppercase letter."
    if not re.search(r"[a-z]", pwd): return "Password must contain at least one lowercase letter."
    if not re.search(r"\d", pwd): return "Password must contain at least one number."
    if not re.search(r"[@$!%*?&_]", pwd): return "Password must contain at least one special symbol (@, $, !, %, *, ?, &, _)."
    return None
def get_email_error(email):
    if not re.match(r"^[a-zA-Z]{2,}.*@[a-zA-Z]{2,}\.[a-zA-Z]{2,}$", email):
        return "Email must have at least 2 letters before '@', 2 letters between '@' and '.', and 2 letters after '.' (e.g., ab@cd.ef)."
    return None
def get_username_error(uname):
    if len(uname) < 3:
        return "Username must be at least 3 characters long."
    if not re.match(r"^[a-zA-Z0-9_.-]+$", uname):
        return "Username can only contain letters, numbers, underscores, dots, and hyphens (no spaces)."
    return None
# --- 7. OTP EMAIL HELPERS ---
def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"
# [FIX #2 & #3] Replaced deprecated utcnow() with timezone-aware now(UTC).
def make_otp_token(email, otp): return jwt.encode({"sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp", "exp": datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}, JWT_SECRET, algorithm="HS256")
def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp": return False, "Token mismatch."
        if check_txt(input_otp, payload["otp_hash"]): return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except: return False, "OTP expired or invalid."
def send_professional_email(to_email, otp, app_pass):
    msg = MIMEMultipart('alternative')
    msg['Message-ID'] = make_msgid()
    msg['Date'] = formatdate(localtime=True)
    msg['From'] = f"Freight Quote Generator <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Your Secure Authentication Code"
    body = f"Hello,\n\nYour secure authentication code for the Intelligent Freight Quote Generator is: {otp}\n\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes.\nIf you did not request this, please ignore this email."
    msg.attach(MIMEText(body, 'plain'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587); s.starttls(); s.login(SENDER_EMAIL, app_pass); s.sendmail(SENDER_EMAIL, to_email, msg.as_string()); s.quit()
        return True, "Email sent successfully!"
    except Exception as e: return False, f"SMTP Error: {str(e)}"
def auth_header(title, sub="Intelligent Analytics Network"):
    st.markdown(f"<div style='text-align:center;padding:0 0 2rem;'><div style='font-size:42px;margin-bottom:8px;'>🚢</div><h1 style='margin:0;font-size:2rem;font-weight:700;'>Intelligent Freight Hub</h1><p style='color:{COLORS['muted']};font-size:14px;'>{sub}</p><div style='margin-top:1.5rem;font-weight:500;font-size:1.1rem;color:{COLORS['accent']};'>{title}</div></div>", unsafe_allow_html=True)
# [FIX #4] Helper to clean up all reset-related session state in one place.
def clear_reset_state():
    st.session_state.reset_mode = None
    st.session_state.reset_email = None
    st.session_state.jwt_otp_token = ""
    st.session_state.pop("sq_p", None)
# ============================================================
# MAIN APPLICATION ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]: st.session_state.page = "Login"
    _, mid, _ = st.columns([1, 1.2, 1])
    with mid:
        # --- LOGIN (Generic Errors) ---
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            identifier = st.text_input("Username or Email", placeholder="Enter your username or email").strip()
            pwd = st.text_input("Password", type="password", placeholder="Enter your password")
            st.markdown("<br>", unsafe_allow_html=True)
            col_l, col_c, col_r = st.columns([1, 1, 1])
            if col_l.button("Sign In →", use_container_width=True):
                if not identifier or not pwd: st.error("⚠️ Username/Email and Password are required.")
                else:
                    with get_db() as c: r = c.execute("SELECT email, password_hash FROM users WHERE email=? OR username=?", (identifier.lower(), identifier)).fetchone()
                    if r and check_txt(pwd, r[1]):
                        st.session_state.token = make_jwt(r[0]); navigate("Dashboard")
                    else:
                        st.error("❌ Invalid credentials. Please try again.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")
        # --- SIGNUP (Highly Specific Errors) ---
        elif st.session_state.page == "Signup":
            auth_header("Create an account")
            uname = st.text_input("Username", placeholder="e.g. JohnDoe123").strip()
            email = st.text_input("Email address", placeholder="e.g. john@example.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min 8 chars, 1 Uppercase, 1 Number, 1 Symbol")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Enter your security answer").strip()
            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("Create Account & Login →", use_container_width=True):
                email_err = get_email_error(email)
                pwd_err = get_password_error(pwd)
                uname_err = get_username_error(uname)
                if not uname: st.error("⚠️ Username is required.")
                elif uname_err: st.error(f"❌ {uname_err}")
                elif not email: st.error("⚠️ Email address is required.")
                elif not pwd: st.error("⚠️ Password is required.")
                elif not confirm_pwd: st.error("⚠️ Please confirm your password.")
                elif not sa: st.error("⚠️ Security answer is required.")
                elif email_err: st.error(f"❌ {email_err}")
                elif pwd_err: st.error(f"❌ {pwd_err}")
                elif pwd != confirm_pwd: st.error("❌ Passwords do not match. Please retype them carefully.")
                else:
                    success = False
                    with get_db() as c:
                        existing_user = c.execute("SELECT username, email FROM users WHERE username=? OR email=?", (uname, email)).fetchone()
                        if existing_user:
                            if existing_user[0].lower() == uname.lower():
                                st.error(f"❌ The username '{uname}' is already taken. Please choose another one.")
                            else:
                                st.error(f"❌ The email '{email}' is already registered. Please sign in instead.")
                        else:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower())))
                            c.commit()
                            success = True
                    if success:
                        st.session_state.token = make_jwt(email)
                        navigate("Dashboard")
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")
        # --- FORGOT PASSWORD ---
        elif st.session_state.page == "Forgot":
            auth_header("Reset your password")
            if not st.session_state.reset_mode:
                email = st.text_input("Registered email address", placeholder="Enter your registered email ID").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)
                c1, c2 = st.columns(2)
                # Check format before hitting the database
                if c1.button("Via Security Question", use_container_width=True):
                    email_err = get_email_error(email)
                    if not email: st.error("⚠️ Please enter your registered email address first.")
                    elif email_err: st.error(f"❌ {email_err}")
                    else:
                        with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                        if r: st.session_state.reset_email = email; st.session_state.sq_p = r[0]; st.session_state.reset_mode = "sq"; st.rerun()
                        else: st.error("❌ No account found with that email address.")
                # Check format before hitting the database
                if c2.button("Via OTP", use_container_width=True):
                    email_err = get_email_error(email)
                    if not email: st.error("⚠️ Please enter your registered email address first.")
                    elif email_err: st.error(f"❌ {email_err}")
                    else:
                        with get_db() as c: r = c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone()
                        if r:
                            # --- OTP Cooldown Check (60 seconds) ---
                            current_time = time.time()
                            if current_time - st.session_state.last_otp_time < 60:
                                wait_time = int(60 - (current_time - st.session_state.last_otp_time))
                                st.error(f"⏳ Please wait {wait_time} seconds before requesting another OTP.")
                            else:
                                otp = generate_otp()
                                with st.spinner("Sending OTP..."): ok, msg = send_professional_email(email, otp, EMAIL_PASSWORD)
                                # [FIX #1] Fixed indentation: this if/else belongs to the send result,
                                # and the success branch now stores the OTP token and navigates properly.
                                if ok:
                                    st.session_state.last_otp_time = current_time
                                    st.session_state.reset_email = email
                                    st.session_state.jwt_otp_token = make_otp_token(email, otp)
                                    st.session_state.reset_mode = "otp"
                                    st.rerun()
                                else:
                                    st.error(f"❌ Transmission Failed: {msg}")
                        else: st.error("❌ No account found with that email address.")
            elif st.session_state.reset_mode == "sq":
                st.info(f"❓ Security Question: **{st.session_state.sq_p}**")
                ans = st.text_input("Your answer", placeholder="Enter answer here").lower().strip()
                npw = st.text_input("New password", type="password", placeholder="Enter new password")
                confirm_npw = st.text_input("Confirm new password", type="password", placeholder="Re-enter new password")
                if st.button("Reset Password →", use_container_width=True):
                    pwd_err = get_password_error(npw)
                    if not ans: st.error("⚠️ Security answer is required.")
                    elif not npw: st.error("⚠️ New password is required.")
                    elif not confirm_npw: st.error("⚠️ Please confirm your new password.")
                    elif pwd_err: st.error(f"❌ {pwd_err}")
                    elif npw != confirm_npw: st.error("❌ New passwords do not match.")
                    else:
                        success = False
                        with get_db() as c:
                            r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
                                c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                c.commit()
                                success = True
                            else:
                                st.error("❌ Incorrect security answer. Please try again.")
                        if success:
                            st.success("✅ Password updated successfully.")
                            time.sleep(1)
                            # [FIX #4] Clean up all reset state including sq_p.
                            clear_reset_state()
                            navigate("Login")
            elif st.session_state.reset_mode == "otp":
                st.info(f"📧 Code sent to {st.session_state.reset_email}")
                otp_input = st.text_input("6-Digit Verification Code", max_chars=6, placeholder="e.g. 123456").strip()
                npw = st.text_input("New Password", type="password", placeholder="Enter new password")
                confirm_npw = st.text_input("Confirm New Password", type="password", placeholder="Re-enter new password")
                if st.button("Verify & Update Password →", use_container_width=True):
                    pwd_err = get_password_error(npw)
                    if not otp_input: st.error("⚠️ Verification code is required.")
                    # [FIX #6] Added digit-only validation so "abcdef" doesn't slip through.
                    elif not otp_input.isdigit(): st.error("⚠️ Verification code must contain only digits.")
                    elif len(otp_input) != 6: st.error("⚠️ Verification code must be exactly 6 digits.")
                    elif not npw: st.error("⚠️ New password is required.")
                    elif not confirm_npw: st.error("⚠️ Please confirm your new password.")
                    elif pwd_err: st.error(f"❌ {pwd_err}")
                    elif npw != confirm_npw: st.error("❌ New passwords do not match.")
                    else:
                        ok, msg = verify_otp_token(st.session_state.jwt_otp_token, otp_input, st.session_state.reset_email)
                        if ok:
                            with get_db() as c:
                                c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                c.commit()
                            st.success("✅ Password updated successfully.")
                            time.sleep(1)
                            # [FIX #4] Clean up all reset state.
                            clear_reset_state()
                            navigate("Login")
                        else: st.error(f"❌ {msg}")
            # [FIX #4] Cancel button now cleans up all stale state.
            if st.button("← Cancel", use_container_width=True):
                clear_reset_state()
                navigate("Login")
# ============================================================
# DASHBOARDS
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    # Graceful fallback if token is invalid
    if not payload:
        st.session_state.token = None
        navigate("Login")
    email = payload["email"]
    # Safety Check: Handles cases where user was deleted or DB was reset
    with get_db() as c:
        user_record = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()
    if not user_record:
        st.session_state.token = None
        navigate("Login")
    else:
        uname = user_record[0]
        col1, col2 = st.columns([5, 1])
        with col1:
            st.markdown(f"<h1 style='color:{COLORS['text']};margin:0;font-size:28px;'>Analytics Dashboard</h1>", unsafe_allow_html=True)
            st.markdown(f"<div style='color:{COLORS['muted']};font-size:14px;margin-bottom:20px;'>{'🛡️ Administrator Access' if email=='admin@infosys.com' else f'👤 Welcome, {uname}'}</div>", unsafe_allow_html=True)
        with col2:
            if st.button("🚪 Logout", use_container_width=True):
                st.session_state.token = None
                navigate("Login")
        if email == "admin@infosys.com":
            st.markdown(f"""
            <div style="background:{COLORS['card']};border:1px solid {COLORS['border']};border-radius:8px;padding:24px;margin-bottom:24px;">
                <h3 style='margin-top:0;'>🛡️ Admin Dashboard</h3>
                <p style='color:{COLORS['muted']};'>List of all registered users.</p>
            </div>
            """, unsafe_allow_html=True)
            with get_db() as c: users = c.execute("SELECT id, username, email FROM users").fetchall()
            st.table([{"ID": u[0], "Username": u[1], "Email": u[2]} for u in users])
        else:
            c1, c2, c3, c4 = st.columns(4)
            for col, icon, lbl, val in [(c1, "📦", "Active Tasks", "60"), (c2, "📄", "Documents", "207"), (c3, "⚡", "Efficiency", "98.4%"), (c4, "🛡️", "Security", "Active")]:
                col.markdown(f"<div style='background:{COLORS['card']};border:1px solid {COLORS['border']};border-radius:8px;padding:20px;text-align:center;'><div style='font-size:28px;margin-bottom:8px;'>{icon}</div><div style='color:{COLORS['muted']};font-size:12px;text-transform:uppercase;'>{lbl}</div><div style='color:{COLORS['accent']};font-size:20px;font-weight:700;margin-top:4px;'>{val}</div></div>", unsafe_allow_html=True)


In [ ]:
import os
import sys
import time
import subprocess
import signal
# --- 1. Safely retrieve secrets from Colab Vault ---
try:
    from google.colab import userdata
    os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
    os.environ['EMAIL_ADDRESS'] = userdata.get('EMAIL_ADDRESS')
    os.environ['JWT_SECRET'] = userdata.get('JWT_SECRET')
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
except Exception as e:
    print(f"⚠️ Secret Error: {e}")
    print("   Make sure all secrets (EMAIL_PASSWORD, EMAIL_ADDRESS, JWT_SECRET, NGROK_AUTHTOKEN)")
    print("   are added in the Colab Secrets tab and 'Notebook access' is turned ON.")
    ngrok_token = None
# --- 2. Import pyngrok with a clear error if missing ---
try:
    from pyngrok import ngrok, conf as ngrok_conf
except ModuleNotFoundError:
    print("❌ pyngrok is not installed. Run this in a cell first:")
    print("   !pip install pyngrok -q")
    raise SystemExit("Missing dependency: pyngrok")
# --- 3. Set ngrok auth token ---
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
else:
    print("⚠️ NGROK_AUTHTOKEN not found. Tunnel may fail on the free tier.")
# --- 4. Clean up any stuck processes from previous runs ---
# [FIX] Use subprocess instead of !-magic so it works inside try/except reliably.
subprocess.run(["pkill", "-f", "streamlit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "ngrok"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# [FIX] Also disconnect any leftover pyngrok tunnels (free tier allows only 1).
try:
    open_tunnels = ngrok.get_tunnels()
    for t in open_tunnels:
        ngrok.disconnect(t.public_url)
except Exception:
    pass  # No tunnels open, nothing to clean
time.sleep(2)
# --- 5. Start the Streamlit server in the background ---
process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",         # Suppress "You can now view your app" prompt
     "--browser.serverAddress", "0.0.0.0",
     "--server.enableCORS", "false",       # Allow ngrok iframe/redirect
     "--server.enableXsrfProtection", "false"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print("⏳ Waiting for Streamlit to boot...")
time.sleep(5)  # Give the server time to start
# Check that the process actually started
if process.poll() is not None:
    print("❌ Streamlit failed to start. Check your app.py for import errors.")
    print("   Tip: run  !streamlit run app.py  in a cell to see the traceback.")
    raise SystemExit("Streamlit process exited prematurely")
# --- 6. Open the Ngrok Tunnel ---
try:
    # [FIX] pyngrok v7+ returns an NgrokTunnel object; v5 also has .public_url.
    # Wrapping in a try to handle both APIs gracefully.
    tunnel = ngrok.connect(8501)
    # pyngrok v5: tunnel.public_url   |   pyngrok v7: tunnel.public_url (still works)
    public_url = tunnel.public_url
    print("=" * 65)
    print(f"🚀 Intelligent Freight Portal is LIVE!")
    print(f"👉 {public_url}")
    print("=" * 65)
    print("⏳ Server is running — click the link above to open your app.")
    print("   Press the ■ Stop button on this cell to shut everything down.")
    print()
    # Keep the cell alive so the tunnel stays open
    while True:
        # [FIX] Also check if Streamlit is still running; restart message if it died.
        if process.poll() is not None:
            print("⚠️ Streamlit process has stopped unexpectedly.")
            break
        time.sleep(1)
except KeyboardInterrupt:
    # This fires when the user clicks the Stop button on the Colab cell.
    print("\n🛑 Shutting down...")
except Exception as e:
    print(f"⚠️ Ngrok Error: {e}")
    print("   Common fixes:")
    print("   • Make sure NGROK_AUTHTOKEN is set correctly in Colab Secrets.")
    print("   • Free ngrok accounts only allow 1 tunnel. Previous one may still be open.")
    print("   • Try running this cell again — cleanup above should fix stale tunnels.")
finally:
    # [FIX] Always clean up, using subprocess (not !-magic) for reliability.
    print("   Stopping Streamlit...")
    process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
    print("   Closing ngrok tunnel...")
    try:
        ngrok.kill()
    except Exception:
        pass
    # Belt-and-suspenders: also pkill in case of orphaned processes
    subprocess.run(["pkill", "-f", "streamlit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["pkill", "-f", "ngrok"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("✅ All services stopped cleanly.")